# 04. Seq2Seq + Attention

## 학습 목표
- Encoder-Decoder 구조와 정보 병목 문제 이해
- 간단한 Seq2Seq를 PyTorch로 구현 (날짜 형식 변환)
- Bahdanau Attention을 직접 구현하고 시각화

## 핵심 논문
- [Sequence to Sequence Learning with Neural Networks (Sutskever et al., 2014)](https://arxiv.org/abs/1409.3215)
- [Neural Machine Translation by Jointly Learning to Align and Translate (Bahdanau et al., 2015)](https://arxiv.org/abs/1409.0473)

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from torch.utils.data import Dataset, DataLoader

## 1. Encoder-Decoder 구조

### 개념

시퀀스를 입력받아 다른 시퀀스를 출력하는 구조:

```
입력 시퀀스       Encoder         Context Vector         Decoder       출력 시퀀스
[x1, x2, x3] → [RNN/LSTM] → [고정 길이 벡터 c] → [RNN/LSTM] → [y1, y2, y3, y4]
```

- **Encoder**: 입력 시퀀스를 하나의 고정 길이 벡터(context vector)로 압축
- **Decoder**: context vector를 받아 출력 시퀀스를 한 토큰씩 생성

### 활용 분야

| 과제 | 입력 | 출력 |
|------|------|------|
| 기계 번역 | "I love you" | "나는 너를 사랑해" |
| 요약 | 긴 문서 | 짧은 요약 |
| 챗봇 | 질문 | 답변 |
| 날짜 변환 | "January 5, 2021" | "2021-01-05" |

### 정보 병목 문제 (Information Bottleneck)

Encoder의 마지막 hidden state 하나에 **전체 입력 정보를 압축**해야 함.

- 짧은 문장: 잘 동작
- 긴 문장: 앞쪽 정보가 손실됨 → 성능 하락

$$\text{"The cat that the dog chased ran away"} \rightarrow \underbrace{[h_T]}_{\text{이 벡터 하나에 모든 정보?}}$$

→ 이 문제를 해결하기 위해 **Attention**이 등장.

---
## 2. Seq2Seq 구현: 날짜 형식 변환

간단하지만 실용적인 예제: `"January 5, 2021"` → `"2021-01-05"`

### 데이터셋 생성

In [ ]:
# 날짜 데이터셋 생성
MONTHS = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]

def generate_date_pair():
    """랜덤 날짜 쌍 생성: (입력, 출력)"""
    year = random.randint(1950, 2030)
    month = random.randint(1, 12)
    day = random.randint(1, 28)  # 간단히 28일까지
    
    # 입력: "January 5, 2021" (다양한 형식)
    fmt = random.choice([
        f"{MONTHS[month-1]} {day}, {year}",         # January 5, 2021
        f"{day} {MONTHS[month-1]} {year}",           # 5 January 2021
        f"{MONTHS[month-1][:3]} {day}, {year}",     # Jan 5, 2021
    ])
    
    # 출력: "2021-01-05"
    target = f"{year}-{month:02d}-{day:02d}"
    
    return fmt, target

# 데이터셋 생성
random.seed(42)
data_pairs = [generate_date_pair() for _ in range(5000)]

print("데이터 예시:")
for src, tgt in data_pairs[:10]:
    print(f"  '{src}' → '{tgt}'")

In [ ]:
# 문자 단위 어휘 사전
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'  # Start of Sequence
EOS_TOKEN = '<EOS>'  # End of Sequence

# 입력에 사용되는 모든 문자
src_chars = sorted(set(ch for src, _ in data_pairs for ch in src))
src_vocab = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN] + src_chars
src_char2idx = {ch: i for i, ch in enumerate(src_vocab)}
src_idx2char = {i: ch for ch, i in src_char2idx.items()}

# 출력에 사용되는 모든 문자
tgt_chars = sorted(set(ch for _, tgt in data_pairs for ch in tgt))
tgt_vocab = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN] + tgt_chars
tgt_char2idx = {ch: i for i, ch in enumerate(tgt_vocab)}
tgt_idx2char = {i: ch for ch, i in tgt_char2idx.items()}

print(f"입력 어휘 크기: {len(src_vocab)} ({src_vocab[:5]}...)")
print(f"출력 어휘 크기: {len(tgt_vocab)} ({tgt_vocab})")

# 최대 시퀀스 길이
MAX_SRC_LEN = max(len(src) for src, _ in data_pairs) + 2  # +SOS+EOS
MAX_TGT_LEN = max(len(tgt) for _, tgt in data_pairs) + 2
print(f"최대 입력 길이: {MAX_SRC_LEN}, 최대 출력 길이: {MAX_TGT_LEN}")

In [ ]:
# 문자열 → 인덱스 변환
def encode_sequence(text, char2idx, max_len):
    indices = [char2idx[SOS_TOKEN]] + [char2idx[ch] for ch in text] + [char2idx[EOS_TOKEN]]
    # 패딩
    indices += [char2idx[PAD_TOKEN]] * (max_len - len(indices))
    return indices[:max_len]

def decode_sequence(indices, idx2char):
    chars = []
    for idx in indices:
        ch = idx2char[idx]
        if ch == EOS_TOKEN:
            break
        if ch not in [PAD_TOKEN, SOS_TOKEN]:
            chars.append(ch)
    return ''.join(chars)

# Dataset 클래스
class DateDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_ids = encode_sequence(src, src_char2idx, MAX_SRC_LEN)
        tgt_ids = encode_sequence(tgt, tgt_char2idx, MAX_TGT_LEN)
        return torch.LongTensor(src_ids), torch.LongTensor(tgt_ids)

# 학습/검증 분리
train_data = DateDataset(data_pairs[:4000])
val_data = DateDataset(data_pairs[4000:])
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)

print(f"학습 데이터: {len(train_data)}개")
print(f"검증 데이터: {len(val_data)}개")

# 인코딩 확인
src_sample, tgt_sample = train_data[0]
print(f"\n인코딩 예시:")
print(f"  입력: {data_pairs[0][0]}")
print(f"  인덱스: {src_sample[:15].tolist()}...")
print(f"  디코딩: '{decode_sequence(src_sample.tolist(), src_idx2char)}'")

### Encoder 구현

In [ ]:
class Encoder(nn.Module):
    """GRU 기반 Encoder"""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
    
    def forward(self, x):
        """
        x: (batch, src_len)
        returns:
            outputs: (batch, src_len, hidden_dim) - 모든 hidden states
            hidden: (1, batch, hidden_dim) - 마지막 hidden state
        """
        embedded = self.embedding(x)         # (batch, src_len, embed_dim)
        outputs, hidden = self.gru(embedded) # outputs: all hidden states
        return outputs, hidden

# 테스트
EMBED_DIM = 32
HIDDEN_DIM = 64

encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
sample_src, _ = next(iter(train_loader))
enc_outputs, enc_hidden = encoder(sample_src)
print(f"Encoder 출력:")
print(f"  모든 hidden states: {enc_outputs.shape}  (batch, src_len, hidden)")
print(f"  마지막 hidden state: {enc_hidden.shape}  (1, batch, hidden)")

### Decoder 구현 (Attention 없는 버전)

In [ ]:
class DecoderBasic(nn.Module):
    """기본 Decoder (Attention 없음)"""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden):
        """
        x: (batch, 1) - 현재 토큰
        hidden: (1, batch, hidden_dim) - 이전 hidden state
        """
        embedded = self.embedding(x)           # (batch, 1, embed_dim)
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc(output.squeeze(1)) # (batch, vocab_size)
        return prediction, hidden

### Seq2Seq 모델 (Attention 없음)

In [ ]:
class Seq2SeqBasic(nn.Module):
    """기본 Seq2Seq (Attention 없음)"""
    
    def __init__(self, encoder, decoder, tgt_vocab_size):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab_size = tgt_vocab_size
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        src: (batch, src_len)
        tgt: (batch, tgt_len)
        """
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        
        # Encoder
        _, hidden = self.encoder(src)  # context vector = 마지막 hidden
        
        # Decoder 시작 토큰
        dec_input = tgt[:, 0].unsqueeze(1)  # <SOS>
        outputs = torch.zeros(batch_size, tgt_len, self.tgt_vocab_size)
        
        for t in range(1, tgt_len):
            prediction, hidden = self.decoder(dec_input, hidden)
            outputs[:, t] = prediction
            
            # Teacher Forcing: 확률적으로 정답 토큰을 다음 입력으로 사용
            if random.random() < teacher_forcing_ratio:
                dec_input = tgt[:, t].unsqueeze(1)  # 정답 사용
            else:
                dec_input = prediction.argmax(dim=1).unsqueeze(1)  # 예측 사용
        
        return outputs


# 모델 생성
encoder_basic = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
decoder_basic = DecoderBasic(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM)
model_basic = Seq2SeqBasic(encoder_basic, decoder_basic, len(tgt_vocab))

print(f"모델 파라미터 수: {sum(p.numel() for p in model_basic.parameters()):,}")

In [ ]:
# 학습 함수
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        optimizer.zero_grad()
        output = model(src, tgt)  # (batch, tgt_len, vocab)
        
        # (batch * tgt_len, vocab) vs (batch * tgt_len)
        output = output[:, 1:].reshape(-1, output.size(-1))
        target = tgt[:, 1:].reshape(-1)
        
        loss = criterion(output, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for src, tgt in loader:
            output = model(src, tgt, teacher_forcing_ratio=0)
            
            output_flat = output[:, 1:].reshape(-1, output.size(-1))
            target_flat = tgt[:, 1:].reshape(-1)
            loss = criterion(output_flat, target_flat)
            total_loss += loss.item()
            
            # 정확도: 전체 시퀀스가 일치하는 비율
            predicted = output[:, 1:].argmax(dim=-1)
            for i in range(src.size(0)):
                pred_str = decode_sequence(predicted[i].tolist(), tgt_idx2char)
                tgt_str = decode_sequence(tgt[i].tolist(), tgt_idx2char)
                if pred_str == tgt_str:
                    correct += 1
                total += 1
    
    return total_loss / len(loader), correct / total

In [ ]:
# 기본 Seq2Seq 학습
optimizer = optim.Adam(model_basic.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # PAD 무시

EPOCHS = 20
train_losses = []
val_accs = []

print("=== 기본 Seq2Seq (Attention 없음) 학습 ===")
for epoch in range(EPOCHS):
    train_loss = train_epoch(model_basic, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model_basic, val_loader, criterion)
    train_losses.append(train_loss)
    val_accs.append(val_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}, Val Acc: {val_acc:.2%}")

basic_final_acc = val_accs[-1]
print(f"\n최종 정확도: {basic_final_acc:.2%}")

In [ ]:
# 예측 예시
def predict(model, src_text, with_attention=False):
    model.eval()
    src_ids = encode_sequence(src_text, src_char2idx, MAX_SRC_LEN)
    src_tensor = torch.LongTensor([src_ids])
    tgt_ids = [tgt_char2idx[SOS_TOKEN]] + [tgt_char2idx[PAD_TOKEN]] * (MAX_TGT_LEN - 1)
    tgt_tensor = torch.LongTensor([tgt_ids])
    
    with torch.no_grad():
        output = model(src_tensor, tgt_tensor, teacher_forcing_ratio=0)
    
    predicted = output[0, 1:].argmax(dim=-1).tolist()
    return decode_sequence(predicted, tgt_idx2char)

print("=== 기본 Seq2Seq 예측 ===")
test_inputs = [
    "January 5, 2021",
    "March 15, 1999",
    "Dec 25, 2000",
    "7 August 1985",
    "November 30, 2025",
]

for inp in test_inputs:
    pred = predict(model_basic, inp)
    print(f"  '{inp}' → '{pred}'")

---
## 3. Attention 메커니즘

### 왜 필요한가?

기본 Seq2Seq의 문제: Encoder가 전체 입력을 **하나의 고정 벡터**로 압축.

**Attention의 아이디어**: Decoder가 출력 토큰을 생성할 때마다, Encoder의 **모든 hidden state를 참조**하여 관련 있는 부분에 집중(attend).

### Bahdanau Attention 수식

Decoder의 시점 $t$에서:

1. **에너지(energy)** 계산: Decoder hidden과 각 Encoder hidden의 관련성

$$e_{tj} = \mathbf{v}^\top \tanh(\mathbf{W}_1 \mathbf{s}_{t-1} + \mathbf{W}_2 \mathbf{h}_j)$$

  - $\mathbf{s}_{t-1}$: Decoder의 이전 hidden state
  - $\mathbf{h}_j$: Encoder의 j번째 hidden state

2. **Attention 가중치** (softmax로 정규화):

$$\alpha_{tj} = \frac{\exp(e_{tj})}{\sum_k \exp(e_{tk})}$$

3. **Context vector** (가중 합):

$$\mathbf{c}_t = \sum_j \alpha_{tj} \mathbf{h}_j$$

4. 이 $\mathbf{c}_t$를 Decoder의 입력에 결합하여 예측.

---
## 4. Attention 직접 구현

In [ ]:
class BahdanauAttention(nn.Module):
    """Bahdanau (Additive) Attention"""
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)  # Decoder hidden용
        self.W2 = nn.Linear(hidden_dim, hidden_dim)  # Encoder hidden용
        self.v = nn.Linear(hidden_dim, 1)             # 에너지 → 스칼라
    
    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (1, batch, hidden) → (batch, 1, hidden)
        encoder_outputs: (batch, src_len, hidden)
        """
        # decoder_hidden을 src_len 차원으로 확장
        dec_h = decoder_hidden.permute(1, 0, 2)  # (batch, 1, hidden)
        
        # 에너지 계산
        energy = torch.tanh(
            self.W1(dec_h) + self.W2(encoder_outputs)
        )  # (batch, src_len, hidden)
        
        energy = self.v(energy).squeeze(2)  # (batch, src_len)
        
        # Attention 가중치
        attention_weights = F.softmax(energy, dim=1)  # (batch, src_len)
        
        # Context vector: 가중 합
        context = torch.bmm(
            attention_weights.unsqueeze(1),  # (batch, 1, src_len)
            encoder_outputs                   # (batch, src_len, hidden)
        ).squeeze(1)  # (batch, hidden)
        
        return context, attention_weights

In [ ]:
class DecoderWithAttention(nn.Module):
    """Attention이 있는 Decoder"""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim)
        # GRU 입력: embed + context
        self.gru = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden, encoder_outputs):
        """
        x: (batch, 1)
        hidden: (1, batch, hidden)
        encoder_outputs: (batch, src_len, hidden)
        """
        embedded = self.embedding(x)  # (batch, 1, embed_dim)
        
        # Attention
        context, attn_weights = self.attention(hidden, encoder_outputs)
        # context: (batch, hidden), attn_weights: (batch, src_len)
        
        # GRU 입력: [embedding; context]
        gru_input = torch.cat([
            embedded,
            context.unsqueeze(1)  # (batch, 1, hidden)
        ], dim=2)  # (batch, 1, embed_dim + hidden)
        
        output, hidden = self.gru(gru_input, hidden)
        prediction = self.fc(output.squeeze(1))  # (batch, vocab_size)
        
        return prediction, hidden, attn_weights

In [ ]:
class Seq2SeqAttention(nn.Module):
    """Seq2Seq with Bahdanau Attention"""
    
    def __init__(self, encoder, decoder, tgt_vocab_size):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab_size = tgt_vocab_size
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        
        # Encoder: 모든 hidden states 사용 (Attention을 위해)
        encoder_outputs, hidden = self.encoder(src)
        
        dec_input = tgt[:, 0].unsqueeze(1)
        outputs = torch.zeros(batch_size, tgt_len, self.tgt_vocab_size)
        attentions = torch.zeros(batch_size, tgt_len, src.size(1))
        
        for t in range(1, tgt_len):
            prediction, hidden, attn_weights = self.decoder(
                dec_input, hidden, encoder_outputs
            )
            outputs[:, t] = prediction
            attentions[:, t] = attn_weights
            
            if random.random() < teacher_forcing_ratio:
                dec_input = tgt[:, t].unsqueeze(1)
            else:
                dec_input = prediction.argmax(dim=1).unsqueeze(1)
        
        return outputs, attentions


# Attention 모델 생성
encoder_attn = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
decoder_attn = DecoderWithAttention(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM)
model_attn = Seq2SeqAttention(encoder_attn, decoder_attn, len(tgt_vocab))

print(f"Attention 모델 파라미터 수: {sum(p.numel() for p in model_attn.parameters()):,}")

In [ ]:
# Attention 모델용 학습/평가 함수
def train_epoch_attn(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        optimizer.zero_grad()
        output, _ = model(src, tgt)
        output = output[:, 1:].reshape(-1, output.size(-1))
        target = tgt[:, 1:].reshape(-1)
        loss = criterion(output, target)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_attn(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for src, tgt in loader:
            output, _ = model(src, tgt, teacher_forcing_ratio=0)
            output_flat = output[:, 1:].reshape(-1, output.size(-1))
            target_flat = tgt[:, 1:].reshape(-1)
            loss = criterion(output_flat, target_flat)
            total_loss += loss.item()
            
            predicted = output[:, 1:].argmax(dim=-1)
            for i in range(src.size(0)):
                pred_str = decode_sequence(predicted[i].tolist(), tgt_idx2char)
                tgt_str = decode_sequence(tgt[i].tolist(), tgt_idx2char)
                if pred_str == tgt_str:
                    correct += 1
                total += 1
    return total_loss / len(loader), correct / total

In [ ]:
# Attention 모델 학습
optimizer_attn = optim.Adam(model_attn.parameters(), lr=0.001)

train_losses_attn = []
val_accs_attn = []

print("=== Seq2Seq with Attention 학습 ===")
for epoch in range(EPOCHS):
    train_loss = train_epoch_attn(model_attn, train_loader, optimizer_attn, criterion)
    val_loss, val_acc = evaluate_attn(model_attn, val_loader, criterion)
    train_losses_attn.append(train_loss)
    val_accs_attn.append(val_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss:.4f}, Val Acc: {val_acc:.2%}")

attn_final_acc = val_accs_attn[-1]
print(f"\n최종 정확도: {attn_final_acc:.2%}")

In [ ]:
# 학습 결과 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Loss 비교
ax = axes[0]
ax.plot(train_losses, label='Basic Seq2Seq', color='blue')
ax.plot(train_losses_attn, label='Seq2Seq + Attention', color='red')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 정확도 비교
ax = axes[1]
ax.plot(val_accs, label=f'Basic ({basic_final_acc:.1%})', color='blue')
ax.plot(val_accs_attn, label=f'+ Attention ({attn_final_acc:.1%})', color='red')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Validation Accuracy Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Attention Weight 시각화

Attention의 가장 큰 장점: **모델이 어디에 집중하는지 시각화** 가능.

In [ ]:
def predict_with_attention(model, src_text):
    """Attention weights와 함께 예측"""
    model.eval()
    src_ids = encode_sequence(src_text, src_char2idx, MAX_SRC_LEN)
    src_tensor = torch.LongTensor([src_ids])
    tgt_ids = [tgt_char2idx[SOS_TOKEN]] + [tgt_char2idx[PAD_TOKEN]] * (MAX_TGT_LEN - 1)
    tgt_tensor = torch.LongTensor([tgt_ids])
    
    with torch.no_grad():
        output, attentions = model(src_tensor, tgt_tensor, teacher_forcing_ratio=0)
    
    predicted = output[0, 1:].argmax(dim=-1).tolist()
    pred_text = decode_sequence(predicted, tgt_idx2char)
    
    # Attention weights 정리
    # 유효한 부분만 추출
    src_len = len(src_text) + 2  # +SOS+EOS
    tgt_len = len(pred_text) + 1  # 예측된 길이
    attn_matrix = attentions[0, 1:tgt_len+1, :src_len].numpy()
    
    return pred_text, attn_matrix


def plot_attention(src_text, pred_text, attn_matrix):
    """Attention heatmap 시각화"""
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # 입력/출력 문자 리스트
    src_chars = ['<S>'] + list(src_text) + ['<E>']
    tgt_chars = list(pred_text)
    
    # 행렬 크기 맞추기
    attn = attn_matrix[:len(tgt_chars), :len(src_chars)]
    
    im = ax.imshow(attn, cmap='Blues', aspect='auto')
    
    ax.set_xticks(range(len(src_chars)))
    ax.set_xticklabels(src_chars, fontsize=10)
    ax.set_yticks(range(len(tgt_chars)))
    ax.set_yticklabels(tgt_chars, fontsize=10)
    
    ax.set_xlabel('Input (source)')
    ax.set_ylabel('Output (target)')
    ax.set_title(f"Attention Weights\n'{src_text}' → '{pred_text}'")
    
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()


# 테스트
test_inputs = [
    "January 5, 2021",
    "March 15, 1999",
    "7 August 1985",
]

for inp in test_inputs:
    pred, attn = predict_with_attention(model_attn, inp)
    print(f"'{inp}' → '{pred}'")
    plot_attention(inp, pred, attn)

### Attention 시각화 해석

Heatmap에서 관찰할 수 있는 패턴:

- **연도 출력 시**: 입력의 숫자 부분("2021")에 높은 가중치
- **월 출력 시**: 입력의 월 이름("January")에 높은 가중치
- **일 출력 시**: 입력의 날짜 숫자("5")에 높은 가중치

→ Decoder가 각 출력 토큰을 생성할 때 **입력의 관련 부분에 집중**하고 있음을 확인.

---
## 6. Transformer로의 전환 예고

### Seq2Seq + Attention의 남은 한계

| 한계 | 설명 |
|------|------|
| 순차 처리 | RNN은 시퀀스를 순서대로 처리 → 병렬화 불가 → 느림 |
| 장거리 의존성 | RNN은 여전히 먼 거리의 정보 전달에 약함 |
| Attention의 확장 | Encoder-Decoder 간 Attention만 사용 (Self-Attention 없음) |

### "Attention Is All You Need" (Vaswani et al., 2017)

RNN을 완전히 제거하고, **Attention만으로** Encoder-Decoder를 구성:

```
Seq2Seq + RNN + Attention  →  Transformer (Attention only)
```

#### Transformer의 핵심 구성요소 (예고)

1. **Self-Attention**: 입력 시퀀스 내에서 각 토큰이 다른 토큰에 집중
2. **Multi-Head Attention**: 여러 관점에서 동시에 Attention
3. **Positional Encoding**: 순서 정보를 벡터로 추가 (RNN 없으므로)
4. **Feed-Forward Network**: 각 위치별 비선형 변환

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

> 이 수식은 다음 챕터 (04-transformer)에서 자세히 다룹니다.

In [ ]:
# Seq2Seq → Transformer 진화를 한눈에 정리
print("=== NLP 모델 진화 과정 ===")
print()
evolution = [
    ("BoW/TF-IDF", "단어 순서 무시", "01-text-preprocessing"),
    ("Word2Vec/GloVe", "의미 임베딩, 문맥 고정", "02-word-embeddings"),
    ("RNN/LSTM", "순서 처리 가능, 기울기 소실", "(이전 챕터)"),
    ("Seq2Seq", "시퀀스→시퀀스, 정보 병목", "이 노트북 (Section 2)"),
    ("+ Attention", "입력 전체 참조, 해석 가능", "이 노트북 (Section 3-5)"),
    ("Transformer", "완전 병렬화, Self-Attention", "다음 챕터"),
    ("BERT, GPT", "사전학습 + 미세조정", "이후 챕터"),
]

print(f"{'모델':<20} {'특징':<35} {'학습 위치'}")
print("-" * 75)
for model, feature, location in evolution:
    print(f"{model:<20} {feature:<35} {location}")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Luong Attention 구현

Bahdanau Attention(additive)과 다른 방식인 Luong Attention(multiplicative)을 구현하세요.

$$e_{tj} = \mathbf{s}_t^\top \mathbf{W} \mathbf{h}_j$$

- Bahdanau: $\mathbf{v}^\top \tanh(\mathbf{W}_1 \mathbf{s} + \mathbf{W}_2 \mathbf{h})$ (additive)
- Luong: $\mathbf{s}^\top \mathbf{W} \mathbf{h}$ (multiplicative, 더 간단)

In [ ]:
# TODO: Luong Attention을 구현하세요
#
# class LuongAttention(nn.Module):
#     def __init__(self, hidden_dim):
#         super().__init__()
#         self.W = nn.Linear(hidden_dim, hidden_dim)
#
#     def forward(self, decoder_hidden, encoder_outputs):
#         # decoder_hidden: (1, batch, hidden)
#         # encoder_outputs: (batch, src_len, hidden)
#         #
#         # 1. energy = s^T W h 계산
#         # 2. attention_weights = softmax(energy)
#         # 3. context = weighted sum
#         #
#         # return context, attention_weights
#
# 구현 후:
# 1. DecoderWithAttention에서 BahdanauAttention 대신 LuongAttention을 사용하세요
# 2. 학습 후 Bahdanau와 정확도를 비교하세요
# 3. Attention heatmap이 어떻게 다른지 비교하세요


### 연습 2: Bidirectional Encoder

Encoder를 양방향(Bidirectional) GRU로 바꿔서 성능이 향상되는지 확인하세요.

In [ ]:
# TODO: Bidirectional Encoder를 구현하세요
#
# class BiEncoder(nn.Module):
#     def __init__(self, vocab_size, embed_dim, hidden_dim):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_dim)
#         self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
#         # 양방향 → hidden이 2배 → 원래 크기로 줄이기 위한 Linear
#         self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
#
# 주의사항:
# 1. bidirectional=True → outputs의 hidden_dim이 2배 (순방향 + 역방향)
# 2. hidden도 (2, batch, hidden) → Decoder에 전달하려면 합치거나 변환 필요
# 3. Attention의 encoder_outputs도 hidden_dim*2가 되므로 Attention 모듈도 수정 필요
#
# 학습 후 단방향 Encoder와 정확도를 비교하세요.


---
## 핵심 정리

| 개념 | 설명 | ML에서의 역할 |
|------|------|---------------|
| Encoder-Decoder | 시퀀스 → 고정 벡터 → 시퀀스 | 번역, 요약, 챗봇 |
| 정보 병목 | 고정 벡터에 모든 정보 압축 | 긴 시퀀스에서 성능 하락 원인 |
| Bahdanau Attention | Decoder가 Encoder 전체를 참조 | 정보 병목 해결, 해석 가능 |
| Teacher Forcing | 학습 시 정답을 입력으로 사용 | 학습 안정화, 수렴 가속 |
| Attention Heatmap | 어디에 집중하는지 시각화 | 모델 해석, 디버깅 |
| Transformer 예고 | RNN 제거, Attention만 사용 | 현대 NLP의 기반 (BERT, GPT) |

**다음 챕터**: 04-transformer - Transformer 아키텍처 ("Attention Is All You Need")